# 3 insights accionables para el equipo comercial

**Dataset:** [Pharma Sales Data](https://www.kaggle.com/datasets/milanzdravkovic/pharma-sales-data) — Milan Zdravkovic (2014–2019, ~600k transacciones POS, 8 categorías ATC).

**Objetivo del notebook:** traducir los datos en decisiones comerciales concretas. No es un EDA exhaustivo: seleccionamos los 3 hallazgos con mayor impacto y los acompañamos de una recomendación accionable.

**Estructura:**

1. Contexto y carga de datos
2. Control de calidad (reconciliación entre granularidades)
3. **Insight #1** — Estacionalidad intra-semanal: cuándo vender más
4. **Insight #2** — Ley de Pareto: dónde concentrar el foco
5. **Insight #3** — Crecimiento diferencial YoY: qué categorías rotar
6. Recomendaciones finales


## 1. Contexto y carga de datos

Los 4 CSV originales contienen las mismas ventas con distinta granularidad temporal (horaria, diaria, semanal, mensual). En este notebook trabajamos principalmente con el **diario** y el **horario**, y usamos el mensual para el cálculo YoY.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

# Permite importar ``src`` desde /notebooks.
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analytics import (
    detectar_desvios,
    estacionalidad_hora_dow,
    kpis_por_categoria,
    pareto_productos,
    tendencia_mensual,
    yoy_por_categoria,
)
from src.data_loader import (
    ATC_CATEGORIES,
    DatasetPaths,
    load_daily,
    load_hourly,
    load_monthly,
    quality_report,
    reconcile_granularities,
)

pd.set_option("display.float_format", "{:,.2f}".format)
RAW = ROOT / "data" / "raw"

paths = DatasetPaths.from_dir(RAW)
daily = load_daily(paths.daily)
hourly = load_hourly(paths.hourly)
monthly = load_monthly(paths.monthly)

print(f"Diario:   {len(daily):>6,} filas  ·  {daily['date'].min().date()} → {daily['date'].max().date()}")
print(f"Horario:  {len(hourly):>6,} filas")
print(f"Mensual:  {len(monthly):>6,} filas")

Diario:    2,106 filas  ·  2014-01-02 → 2019-10-08
Horario:  50,532 filas
Mensual:      70 filas


## 2. Control de calidad

Antes de analizar, validamos dos cosas:

1. **Nulos y rangos** por categoría.
2. **Reconciliación cruzada**: al reagregar el diario a mensual, los totales deberían coincidir con el CSV mensual. Si no coinciden, hay un error de ingesta en el dataset que hay que reportar.

In [2]:
quality_report(daily, "diario")

,nulos,min,max,media
ATC (diario),,,,
M01AB,0,0.00,17.34,5.03
M01AE,0,0.00,14.46,3.90
N02BA,0,0.00,16.00,3.88
N02BE,0,0.00,161.00,29.92
N05B,0,0.00,54.83,8.85
N05C,0,0.00,9.00,0.59
R03,0,0.00,45.00,5.51
R06,0,0.00,15.00,2.90


In [3]:
rec = reconcile_granularities(daily, monthly, tolerance=0.01)
print(f"Meses OK (delta ≤ 1%): {rec['ok'].sum()} / {len(rec)}")
rec.tail(5)

Meses OK (delta ≤ 1%): 37 / 70


,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,max_delta_rel,ok
period,,,,,,,,,,
2019-06,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,True
2019-07,0.00,0.00,0.00,0.04,0.00,0.00,0.00,0.00,0.04,False
2019-08,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,True
2019-09,0.00,0.00,0.00,0.00,0.00,0.00,0.25,0.00,0.25,False
2019-10,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,True


## 3. Insight #1 — La demanda se concentra en ventanas predecibles

**Hipótesis:** las ventas no se distribuyen uniformemente a lo largo de la semana/día. Si identificamos las ventanas pico, podemos optimizar staffing, promociones y lanzamientos.

In [4]:
heat = estacionalidad_hora_dow(hourly)
fig = px.imshow(
    heat,
    aspect="auto",
    color_continuous_scale="Blues",
    labels={"x": "Hora del día", "y": "Día", "color": "Ventas promedio"},
    title="Ventas promedio por hora y día de la semana",
)
fig.show()

In [5]:
# Top 5 celdas (día × hora) de mayor venta promedio
top_celdas = (
    heat.stack()
    .sort_values(ascending=False)
    .head(5)
    .rename("ventas_promedio")
    .reset_index()
    .rename(columns={"level_0": "dia", "hour": "hora"})
)
top_celdas

,dia,hora,ventas_promedio
0,Dom,12,6.93
1,Dom,11,6.25
2,Sáb,11,6.19
3,Sáb,12,6.12
4,Dom,10,5.97


**Recomendación accionable #1:** concentrar el 70% del esfuerzo comercial (promos, visitas médicas, material POP) en las 5 ventanas día-hora identificadas arriba. Sacar staffing a las ventanas de baja demanda libera presupuesto sin afectar la facturación.

## 4. Insight #2 — Ley de Pareto: dónde concentrar el foco

**Hipótesis:** una minoría de categorías ATC explica la mayoría de las unidades vendidas. Si la regla 80/20 se cumple, el equipo comercial debe priorizar esas categorías para maximizar retorno por hora invertida.

In [6]:
pareto = pareto_productos(daily)
pareto

,categoria,unidades,share_pct,share_acumulado_pct,en_top_80
0,N02BE,"63,005.40",49.38,49.38,True
1,N05B,"18,645.74",14.61,63.99,True
2,R03,"11,608.82",9.10,73.09,True
3,M01AB,"10,600.94",8.31,81.40,False
4,M01AE,"8,204.62",6.43,87.83,False
5,N02BA,"8,172.21",6.40,94.23,False
6,R06,"6,107.82",4.79,99.02,False
7,N05C,"1,249.96",0.98,100.00,False


In [7]:
fig = px.bar(
    pareto,
    x="categoria",
    y="share_pct",
    text="share_pct",
    color="en_top_80",
    color_discrete_map={True: "#2563eb", False: "#cbd5e1"},
    title="Participación % por categoría ATC (ordenado desc.)",
    labels={"share_pct": "% del total", "categoria": "Categoría ATC"},
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

top80 = pareto[pareto["en_top_80"]]
print(
    f"{len(top80)} de {len(pareto)} categorías ATC "
    f"({len(top80)/len(pareto)*100:.0f}%) explican el "
    f"{top80['share_pct'].sum():.1f}% de las ventas."
)

3 de 8 categorías ATC (38%) explican el 73.1% de las ventas.


**Recomendación accionable #2:** tratar a las categorías marcadas en azul como *core portfolio* — asegurar stock, visibilidad y seguimiento semanal. Las categorías grises son candidatas a análisis de rentabilidad: si el margen no compensa el costo de mantenerlas, evaluar racionalización.

## 5. Insight #3 — Crecimiento diferencial: ganadores y perdedores

**Hipótesis:** no todas las categorías evolucionan igual. El mix de ventas cambia año a año. Identificar qué sube y qué baja permite anticipar la rotación del portfolio.

In [8]:
yoy = yoy_por_categoria(monthly)
yoy

,unidades_2018,unidades_2019,delta_abs,delta_pct,descripcion
M01AB,"1,466.93","1,517.27",50.34,3.43,Antiinflamatorios no esteroides — ácido acético
N05C,193.00,196.00,3.00,1.55,Hipnóticos y sedantes
R03,"2,070.00","2,050.00",-20.00,-0.97,Enfermedades obstructivas de vías respiratorias
R06,"1,095.30","1,073.57",-21.73,-1.98,Antihistamínicos sistémicos
M01AE,"1,171.15","1,117.22",-53.92,-4.60,Antiinflamatorios no esteroides — ácido propió...
N02BA,936.90,879.80,-57.10,-6.09,Analgésicos — ácido salicílico y derivados
N02BE,"9,054.58","8,011.62","-1,042.96",-11.52,Analgésicos — pirazolonas y anilidas
N05B,"2,760.00","2,405.60",-354.40,-12.84,Ansiolíticos


In [9]:
yoy_plot = yoy.reset_index(names="categoria")
fig = px.bar(
    yoy_plot,
    x="categoria",
    y="delta_pct",
    color="delta_pct",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    title="Crecimiento YoY por categoría ATC (último año vs. anterior)",
    labels={"delta_pct": "% variación", "categoria": "Categoría ATC"},
    hover_data=["descripcion"],
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [10]:
# Tendencia mensual agregada — contexto visual
trend = tendencia_mensual(daily)
fig = px.line(
    trend,
    x="date",
    y="total",
    title="Ventas totales mensuales (todas las categorías)",
    labels={"date": "Mes", "total": "Unidades"},
)
fig.show()

**Recomendación accionable #3:** reasignar presupuesto de marketing desde las categorías con delta YoY negativo (rojo) hacia las de crecimiento sostenido (verde). Establecer una revisión trimestral con esta misma métrica como *early warning* del portfolio.

## 6. Recomendaciones consolidadas

| # | Hallazgo | Acción | Métrica de seguimiento |
|---|---|---|---|
| 1 | La demanda se concentra en ventanas día×hora específicas | Reasignar staffing y promos a esas ventanas | Ventas en top-5 celdas / ventas totales |
| 2 | Pocas categorías ATC explican el grueso de las ventas | Priorizar *core portfolio*; revisar rentabilidad de la cola | Share % del top 80 |
| 3 | El mix cambia año a año: hay ganadores y perdedores claros | Reasignar presupuesto marketing según delta YoY | Delta YoY por categoría (trimestral) |

## Próximos pasos (fuera del alcance de este notebook)

- **Automatización del reporte mensual** vía CLI (`python -m pharma_sales monthly-report --month 2019-09`).
- **Alertas automáticas** de desvío (ya implementadas en `analytics.detectar_desvios`) enviadas por mail.
- **Forecasting** simple (regresión lineal + promedio móvil) para proyección del mes siguiente.
- **Tests** unitarios en `tests/` sobre `data_loader` y `analytics`.